# Setup S3 — Descargar CelebA + generar CSV

Corre este notebook **una sola vez** en SageMaker antes de entrenar.

**Qué hace:**
1. Instala dependencias
2. Configura credenciales Kaggle
3. Descarga CelebA (~1.3GB) desde Kaggle a `/tmp/celeba`
4. Genera `celeba_flags_labels.csv` (mapeo flags.txt)
5. Sube todo a `s3://unique-pets/data/celeba/`

**Tiempo estimado:** ~10-15 min (descarga + subida a S3)

In [1]:
import sys
!{sys.executable} -m pip install -q kagglehub

In [2]:
import os, json, shutil, time
import boto3
import sagemaker
import kagglehub
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

BUCKET  = 'unique-pets'
S3_KEY  = 'data/celeba'
TMP_DIR = Path('/tmp/celeba')
TMP_DIR.mkdir(parents=True, exist_ok=True)

session = sagemaker.Session()
s3      = boto3.client('s3')
print(f'Región: {session.boto_region_name}')
print(f'Bucket destino: s3://{BUCKET}/{S3_KEY}/')

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Región: us-east-1
Bucket destino: s3://unique-pets/data/celeba/


## 1. Token Kaggle

Ve a https://www.kaggle.com/settings → **API** → **Create New Token**

In [3]:
import os

KAGGLE_TOKEN = 'KGAT_4bc07b7b10041c2e2b4949a5f38abef1'  # ← pegar tu Access Token de Kaggle

os.environ['KAGGLE_TOKEN'] = KAGGLE_TOKEN
print('Token Kaggle configurado OK')

Token Kaggle configurado OK


## 2. Descargar CelebA desde Kaggle

In [4]:
print('Descargando jessicali9530/celeba-dataset...')
t0 = time.time()

kaggle_path = kagglehub.dataset_download('jessicali9530/celeba-dataset')

print(f'Descargado en {time.time()-t0:.1f}s')
print(f'Path: {kaggle_path}')

for p in sorted(Path(kaggle_path).rglob('*'))[:20]:
    print(p)

Descargando jessicali9530/celeba-dataset...


100%|██████████| 1.33G/1.33G [00:10<00:00, 142MB/s] 

Extracting files...


Descargado en 41.8s
Path: /home/ec2-user/.cache/kagglehub/datasets/jessicali9530/celeba-dataset/versions/2
/home/ec2-user/.cache/kagglehub/datasets/jessicali9530/celeba-dataset/versions/2/img_align_celeba
/home/ec2-user/.cache/kagglehub/datasets/jessicali9530/celeba-dataset/versions/2/img_align_celeba/img_align_celeba
/home/ec2-user/.cache/kagglehub/datasets/jessicali9530/celeba-dataset/versions/2/img_align_celeba/img_align_celeba/000001.jpg
/home/ec2-user/.cache/kagglehub/datasets/jessicali9530/celeba-dataset/versions/2/img_align_celeba/img_align_celeba/000002.jpg
/home/ec2-user/.cache/kagglehub/datasets/jessicali9530/celeba-dataset/versions/2/img_align_celeba/img_align_celeba/000003.jpg
/home/ec2-user/.cache/kagglehub/datasets/jessicali9530/celeba-dataset/versions/2/img_align_celeba/img_align_celeba/000004.jpg
/home/ec2-user/.cache/kagglehub/datasets/jessicali9530/celeba-dataset/versions/2/img_align_celeba/img_align_celeba/000005.jpg
/home/ec2-user/.cache/kagglehub/datasets/jessicali

In [5]:
kaggle_root = Path(kaggle_path)

# Kaggle descarga con estructura anidada: img_align_celeba/img_align_celeba/
# Tomar el directorio más profundo que tenga .jpg
img_dirs   = list(kaggle_root.rglob('img_align_celeba'))
attr_files = list(kaggle_root.rglob('list_attr_celeba*'))
eval_files = list(kaggle_root.rglob('list_eval_partition*'))

# El dir real de imágenes es el que contiene .jpg directamente
IMG_SRC  = next((d for d in sorted(img_dirs, key=lambda x: len(x.parts), reverse=True)
                 if list(d.glob('*.jpg'))), None)
ATTR_SRC = attr_files[0] if attr_files else None
EVAL_SRC = eval_files[0] if eval_files else None

print(f'IMG_SRC:  {IMG_SRC}')
print(f'ATTR_SRC: {ATTR_SRC}')
print(f'EVAL_SRC: {EVAL_SRC}')
if IMG_SRC:
    print(f'JPGs encontrados: {len(list(IMG_SRC.glob("*.jpg"))):,}')

img_align_celeba: [PosixPath('/home/ec2-user/.cache/kagglehub/datasets/jessicali9530/celeba-dataset/versions/2/img_align_celeba'), PosixPath('/home/ec2-user/.cache/kagglehub/datasets/jessicali9530/celeba-dataset/versions/2/img_align_celeba/img_align_celeba')]
list_attr:        [PosixPath('/home/ec2-user/.cache/kagglehub/datasets/jessicali9530/celeba-dataset/versions/2/list_attr_celeba.csv')]
list_eval:        [PosixPath('/home/ec2-user/.cache/kagglehub/datasets/jessicali9530/celeba-dataset/versions/2/list_eval_partition.csv')]


## 3. Generar celeba_flags_labels.csv

In [6]:
# Cargar atributos CelebA
df = pd.read_csv(ATTR_SRC)
attr_cols = [c for c in df.columns if c != 'image_id']
df[attr_cols] = (df[attr_cols] == 1).astype(int)
print(f'CelebA cargado: {df.shape}')

out = pd.DataFrame({'image_id': df['image_id']})

def map_color_pelo(r):
    if r['Bald']:        return 'calvo'
    if r['Black_Hair']:  return 'negro'
    if r['Blond_Hair']:  return 'rubio'
    if r['Brown_Hair']:  return 'castano'
    if r['Gray_Hair']:   return 'gris'
    return 'negro'

def map_textura_pelo(r):
    if r['Straight_Hair']: return 'liso'
    if r['Wavy_Hair']:     return 'ondulado'
    return 'liso'

def map_longitud_pelo(r):
    if r['Bald']:  return 'calvo'
    if r['Bangs']: return 'largo'
    return 'medio'

def map_cejas(r):
    if r['Arched_Eyebrows']: return 'arqueadas'
    if r['Bushy_Eyebrows']:  return 'pobladas'
    return 'normales'

def map_forma_ojos(r):
    if r['Narrow_Eyes']:     return 'rasgada'
    if r['Bags_Under_Eyes']: return 'caida'
    return 'almendrada'

def map_tamano_nariz(r):
    if r['Big_Nose']:    return 'grande'
    if r['Pointy_Nose']: return 'pequena'
    return 'mediana'

def map_forma_nariz(r):
    if r['Big_Nose']:    return 'ancha'
    if r['Pointy_Nose']: return 'respingona'
    return 'recta'

def map_grosor_labios(r):
    return 'carnosos' if r['Big_Lips'] else 'medianos'

def map_pomulos(r):
    return 'altos' if r['High_Cheekbones'] else 'normales'

def map_mandibula(r):
    return 'ancha' if r['Chubby'] else 'suave'

def map_barbilla(r):
    return 'redonda'

def map_forma_cara(r):
    if r['Oval_Face']: return 'oval'
    if r['Chubby']:    return 'redonda'
    return 'oval'

def map_vello_facial(r):
    if r['No_Beard'] and not r['Mustache']: return 'sin_barba'
    if r['Mustache']:                        return 'bigote'
    if r['5_o_Clock_Shadow']:                return 'barba_corta'
    if r['Goatee'] or r['Sideburns']:        return 'barba_corta'
    return 'sin_barba'

def map_tono_piel(r):
    return 'muy_claro' if r['Pale_Skin'] else 'medio'

def map_rango_edad(r):
    return 'joven' if r['Young'] else 'adulto'

print('Aplicando mapeos...')
out['color_pelo']    = df.apply(map_color_pelo,    axis=1)
out['textura_pelo']  = df.apply(map_textura_pelo,  axis=1)
out['longitud_pelo'] = df.apply(map_longitud_pelo, axis=1)
out['cejas']         = df.apply(map_cejas,         axis=1)
out['forma_ojos']    = df.apply(map_forma_ojos,    axis=1)
out['tamano_nariz']  = df.apply(map_tamano_nariz,  axis=1)
out['forma_nariz']   = df.apply(map_forma_nariz,   axis=1)
out['grosor_labios'] = df.apply(map_grosor_labios, axis=1)
out['pomulos']       = df.apply(map_pomulos,       axis=1)
out['mandibula']     = df.apply(map_mandibula,     axis=1)
out['barbilla']      = df.apply(map_barbilla,      axis=1)
out['forma_cara']    = df.apply(map_forma_cara,    axis=1)
out['vello_facial']  = df.apply(map_vello_facial,  axis=1)
out['gafas']         = df['Eyeglasses'].astype(bool)
out['pecas']         = False
out['tono_piel']     = df.apply(map_tono_piel,     axis=1)
out['rango_edad']    = df.apply(map_rango_edad,    axis=1)

csv_path = TMP_DIR / 'celeba_flags_labels.csv'
out.to_csv(csv_path, index=False)
print(f'CSV guardado: {csv_path}  ({len(out):,} filas)')

CelebA cargado: (202599, 41)
Aplicando mapeos...
CSV guardado: /tmp/celeba/celeba_flags_labels.csv  (202,599 filas)


## 4. Subir todo a S3

In [7]:
# Subir list_attr, list_eval y CSV
print('Subiendo archivos de metadatos a S3...')
for local_file, s3_name in [
    (ATTR_SRC,  'list_attr_celeba.txt'),
    (EVAL_SRC,  'list_eval_partition.txt'),
    (csv_path,  'celeba_flags_labels.csv'),
]:
    if local_file and Path(local_file).exists():
        key = f'{S3_KEY}/{s3_name}'
        s3.upload_file(str(local_file), BUCKET, key)
        print(f'  ✓ s3://{BUCKET}/{key}')
    else:
        print(f'  ✗ {s3_name} no encontrado, skip')

Subiendo archivos de metadatos a S3...
  ✓ s3://unique-pets/data/celeba/list_attr_celeba.txt
  ✓ s3://unique-pets/data/celeba/list_eval_partition.txt
  ✓ s3://unique-pets/data/celeba/celeba_flags_labels.csv


In [8]:
# Subir imágenes a S3 (la parte que tarda — ~202K archivos)
if IMG_SRC is None:
    raise FileNotFoundError('No se encontró img_align_celeba en el dataset descargado')

img_files = sorted(Path(IMG_SRC).glob('*.jpg'))
print(f'Imágenes a subir: {len(img_files):,}')

t0 = time.time()
errors = []

for i, img_path in enumerate(tqdm(img_files, desc='Subiendo a S3')):
    key = f'{S3_KEY}/img_align_celeba/{img_path.name}'
    try:
        s3.upload_file(str(img_path), BUCKET, key)
    except Exception as e:
        errors.append((img_path.name, str(e)))

elapsed = time.time() - t0
print(f'\nSubida completa: {len(img_files) - len(errors):,} imágenes en {elapsed/60:.1f} min')
if errors:
    print(f'Errores: {len(errors)}')
    for name, err in errors[:5]:
        print(f'  {name}: {err}')

Imágenes a subir: 0


Subiendo a S3: 0it [00:00, ?it/s]


Subida completa: 0 imágenes en 0.0 min


In [9]:
# Verificar contenido del bucket
result = s3.list_objects_v2(Bucket=BUCKET, Prefix=S3_KEY + '/', MaxKeys=5)
total  = s3.list_objects_v2(Bucket=BUCKET, Prefix=S3_KEY + '/img_align_celeba/')['KeyCount']

print(f's3://{BUCKET}/{S3_KEY}/')
for obj in result.get('Contents', []):
    print(f"  {obj['Key']}  ({obj['Size']/1e6:.1f} MB)")
print(f'  ... ({total:,} imágenes en img_align_celeba/)')
print('\nSetup completo. Ya puedes correr 03_sagemaker_vit_training.ipynb')

s3://unique-pets/data/celeba/
  data/celeba/  (0.0 MB)
  data/celeba/celeba_flags_labels.csv  (27.1 MB)
  data/celeba/img_align_celeba/  (0.0 MB)
  data/celeba/list_attr_celeba.txt  (24.9 MB)
  data/celeba/list_eval_partition.txt  (2.8 MB)
  ... (1 imágenes en img_align_celeba/)

Setup completo. Ya puedes correr 03_sagemaker_vit_training.ipynb
